<a href="https://colab.research.google.com/github/Ape108/FundamentalAnalysisGPT/blob/main/Milestone_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# --- CONFIGURATION ---
TRAIN_MODEL = False  # Change to True when you want to train
FINE_TUNE_MODEL = False
MODEL_SAVE_PATH = "milestone_2_model.pth"
LORA_SAVE_PATH = "lora_adapted_model.pth"

# Environment Setup

In [2]:
# Install required dependencies
!pip install -q numpy "datasets<3" tiktoken

In [3]:
# Import standard libraries
import math
import random
import matplotlib.pyplot as plt
import torch
import torch._dynamo
import datasets

# Suppress dynamo errors for clean output
torch._dynamo.config.suppress_errors = True
# Enable TF32 for faster matmul on Ampere+ GPUs
torch.set_float32_matmul_precision('high')

# Seed for reproducibility
random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## Import Architecture

In [4]:
# Import from your uploaded consolidated architecture file
from architecture import (
    GPTModel,
    create_streaming_dataloaders,
    train_efficient,
    generate_text,
    prepare_data_tiktoken,
    create_dataloaders
)

# Baseline

## Configuration Setup

In [5]:
# Model and Training Configuration
# The "Goldilocks" A100 Configuration
CONFIG = {
    "vocab_size": 0,
    "context_length": 256,    # Kept the same as M2 for a direct apples-to-apples comparison
    "emb_dim": 512,           # A nice step up from 384 for better learning capacity
    "n_heads": 8,             # 512 / 8 = 64 (standard head dimension)
    "n_layers": 6,            # Kept at 6 layers to keep training time very short
    "drop_rate": 0.1,
    "qkv_bias": True,
    "batch_size": 256,
    "learning_rate": 5e-4,
    "max_steps": 2000
}

print("Initial Config:", CONFIG)

Initial Config: {'vocab_size': 0, 'context_length': 256, 'emb_dim': 512, 'n_heads': 8, 'n_layers': 6, 'drop_rate': 0.1, 'qkv_bias': True, 'batch_size': 256, 'learning_rate': 0.0005, 'max_steps': 2000}


## Download and Tokenize Dataset

In [6]:
# Load the full corpus using streaming
dataset_streams = datasets.load_dataset(
    "eloukas/edgar-corpus",
    "full",
    trust_remote_code=True,
    streaming=True
)

train_stream = dataset_streams["train"]
val_stream = dataset_streams["test"]

# Initialize streaming dataloaders
print("Initializing tokenization and dataloaders...")
train_dataloader, val_dataloader, vocab_size, tokenizer = create_streaming_dataloaders(
    train_stream, val_stream, CONFIG
)

# Update config with the dynamic vocabulary size
CONFIG["vocab_size"] = vocab_size
print(f"Updated CONFIG with Vocab Size: {CONFIG}")

Initializing tokenization and dataloaders...
Updated CONFIG with Vocab Size: {'vocab_size': 50257, 'context_length': 256, 'emb_dim': 512, 'n_heads': 8, 'n_layers': 6, 'drop_rate': 0.1, 'qkv_bias': True, 'batch_size': 256, 'learning_rate': 0.0005, 'max_steps': 2000}


## Initialize Model & Optimizer

In [7]:
# Initialize the base generative model
model = GPTModel(CONFIG).to(device)

# Initialize AdamW optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

# Optional: Compile the model for PyTorch 2.0+ speedups (uncomment if desired)
print("Compiling model...")
model = torch.compile(model)

print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters.")

Compiling model...
Model initialized with 70509568 parameters.


## Pretraining Loop

In [8]:
if TRAIN_MODEL:
    EVAL_EVERY = 20

    print("Starting custom generative pretraining...")
    train_losses, val_losses = train_efficient(
        model=model,
        train_loader=train_dataloader,
        val_loader=val_dataloader,
        optimizer=optimizer,
        config=CONFIG,
        device=device,
        accumulation_steps=1,
        eval_every=EVAL_EVERY,
        num_epochs=2,
        max_steps=CONFIG["max_steps"]
    )

    print("Pretraining run complete!")

    # Save the base model weights
    PATH = "milestone_2_model.pth"
    torch.save(model.state_dict(), PATH)
    print("Base model saved to", PATH)
else:
    print("Skipping training. Loading saved weights...")

    # 1. Load the weights into the model
    # (map_location='cpu' ensures it loads safely even if you don't have a GPU active)
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location='cpu'))

    # 2. Set the model to evaluation mode (important for text generation)
    model.eval()
    print("Model loaded successfully!")

Skipping training. Loading saved weights...
Model loaded successfully!


## Plotting & Metrics

In [9]:
if TRAIN_MODEL:
    # Plot the training vs validation loss curve
    steps_index = [i * EVAL_EVERY for i in range(1, len(val_losses) + 1)]

    plt.figure(figsize=(8, 5))
    plt.plot(steps_index, train_losses, label="Train Loss")
    plt.plot(steps_index, val_losses, label="Validation Loss")

    plt.title("Milestone 2: Pretraining Loss Curve")
    plt.xlabel("Training Steps")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.savefig("loss_curve.png")
    plt.show()

    # Calculate and print perplexity
    val_perplexity = math.exp(val_losses[-1])
    train_perplexity = math.exp(train_losses[-1])

    print(f"Final Train Loss: {train_losses[-1]:.2f} | Final Val Loss: {val_losses[-1]:.2f}")
    print(f"Final Train Perplexity: {train_perplexity:.2f} | Final Val Perplexity: {val_perplexity:.2f}")

## Inference Evaluation

In [10]:
# Test generation to evaluate baseline grammatical and structural generation
prompts = [
    "Item 1A. Risk Factors. As a wholesale power trading entity, our financial condition is highly dependent on grid stability and commodity price fluctuations. During periods of extreme weather, unexpected margin calls from our clearing broker could result in",
    "Management's Discussion and Analysis: Regarding the implied volatility skew observed in our equity derivatives portfolio, particularly the out-of-the-money NVDA call options, the primary driver of this pricing anomaly is",
    """Extract the core financial risk from the following paragraph:
Paragraph: The ongoing litigation regarding the delayed deployment of our high-performance computing clusters has severely restricted our operating cash flow, forcing us to draw $50 million from our revolving credit facility.
Core Risk:""",
    "On March 15, the Board of Directors announced the immediate resignation of the Chief Financial Officer. To ensure continuity in our algorithmic trading division and to oversee the upcoming quarterly audit, the Board has appointed",
]

print("--- Base Model Text Generation ---")
for p in prompts:
    input_ids = tokenizer.encode(p)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # Generate 40 tokens per prompt
    output_tensor = generate_text(model, input_tensor, max_new_tokens=100)
    generated_text = tokenizer.decode(output_tensor[0].tolist())

    print(f"\nPrompt: {p}")
    print(f"Output: {generated_text}")

--- Base Model Text Generation ---

Prompt: Item 1A. Risk Factors. As a wholesale power trading entity, our financial condition is highly dependent on grid stability and commodity price fluctuations. During periods of extreme weather, unexpected margin calls from our clearing broker could result in
Output: Item 1A. Risk Factors. As a wholesale power trading entity, our financial condition is highly dependent on grid stability and commodity price fluctuations. During periods of extreme weather, unexpected margin calls from our clearing broker could result in reduced the Company's ability to generate revenue from its operating results in an effort to reduce costs at attractive prices. The Company seeks to enhance its sales and marketing programs with significant capital expenditures under its own control strategy.
The Company believes it can attract high levels of financial resources that are critical to the Company's business. In addition, as well as the consolidation of new systems, th

# Fine-Tuning (LoRA)

## Define LoRA Wrappers

In [11]:
import torch.nn as nn

class LoRALayer(nn.Module):
    """Additive low-rank update for a linear layer."""
    def __init__(self, in_dim, out_dim, rank=8, alpha=8):
        super().__init__()
        self.rank = rank
        self.alpha = alpha

        # A is initialized with Kaiming uniform (standard variance)
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        # B is initialized to zero so the initial LoRA update is exactly 0
        self.B = nn.Parameter(torch.zeros(rank, out_dim))

    def forward(self, x):
        return (self.alpha / self.rank) * (x @ self.A @ self.B)

class LinearWithLoRA(nn.Module):
    """Wraps an existing nn.Linear and adds the LoRA adapter on top."""
    def __init__(self, linear: nn.Linear, rank=8, alpha=8):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank=rank, alpha=alpha)
        self.lora.to(self.linear.weight.device)

    def forward(self, x):
        # Base output + LoRA update
        return self.linear(x) + self.lora(x)

## Inject LoRA into the Model

In [12]:
def replace_linears_with_lora(module, rank=8, alpha=8, name_filter=None, prefix=""):
    """Recursively replaces selected nn.Linear modules with LinearWithLoRA."""
    replaced_names = []

    for name, child in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name

        if isinstance(child, nn.Linear):
            if name_filter is None or name_filter(full_name, child):
                setattr(module, name, LinearWithLoRA(child, rank=rank, alpha=alpha))
                replaced_names.append(full_name)
        else:
            replaced_names.extend(
                replace_linears_with_lora(child, rank=rank, alpha=alpha, name_filter=name_filter, prefix=full_name)
            )
    return replaced_names

def freeze_non_lora_params(model):
    """Freezes the base model, leaving only LoRA A and B parameters trainable."""
    for name, p in model.named_parameters():
        if ".lora.A" in name or ".lora.B" in name:
            p.requires_grad = True
        else:
            p.requires_grad = False

## Apply it to the Base Model

In [13]:
# 1. Load your best base model from Milestone 2
# model.load_state_dict(torch.load("my_base_model.pth"))

# 2. Define which layers to target (Query and Value projections are standard)
def target_attention_layers(full_name, module):
    # UPDATE THESE STRINGS to match your custom model's architecture
    return "Wq" in full_name or "Wv" in full_name

# 3. Inject the adapters
print("Injecting LoRA adapters...")
replaced = replace_linears_with_lora(model, rank=8, alpha=16, name_filter=target_attention_layers)
freeze_non_lora_params(model)

# 4. Prove to the grader that you reduced the parameter count (Rubric requirement!)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable LoRA Parameters: {trainable_params:,}")
print(f"Trainable %: {(trainable_params / total_params) * 100:.3f}%")

Injecting LoRA adapters...
Total Parameters: 70,607,872
Trainable LoRA Parameters: 98,304
Trainable %: 0.139%


## Prepare the FinQA Instruction Dataset

In [14]:
print("Loading FinQA dataset...")
# Load the dataset
dataset = datasets.load_dataset("wandb/finqa-data-processed")

# We'll take a subset for speed, but you can use the whole thing if you have time
train_data = dataset["train"].select(range(5000))
val_data = dataset["test"].select(range(500))

def format_finqa(example):
    """Maps the FinQA columns into a single instruction string for training."""
    prompt = (
        f"Below is an instruction that describes a financial task, paired with an input that provides further context.\n"
        f"Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{example['query']}\n\n"
        f"### Input:\n{example['context']}\n\n"
        f"### Response:\n{example['output']}"
    )
    # Return it under the key 'section_1' so architecture.py can process it natively
    return {"section_1": prompt}

print("Formatting prompts...")
train_data = train_data.map(format_finqa)
val_data = val_data.map(format_finqa)

# Use your existing architecture.py functions!
print("Tokenizing and creating dataloaders...")
train_tokens, val_tokens, encoding, vocab_size = prepare_data_tiktoken(train_data, val_data)

# Ensure your config matches what you used for the base model
config = {
    "vocab_size": vocab_size,
    "context_length": CONFIG['context_length'], # Adjust based on your memory limits
    "batch_size": CONFIG['batch_size'],
}

train_loader, val_loader = create_dataloaders(train_tokens, val_tokens, config, cores=8)
print("Data ready for LoRA Fine-Tuning.")

Loading FinQA dataset...


Repo card metadata block was not found. Setting CardData to empty.


Formatting prompts...
Tokenizing and creating dataloaders...
Tokenizing data using multiprocessing...


Tokenizing Train Data (num_proc=10):   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing Val Data (num_proc=10):   0%|          | 0/500 [00:00<?, ? examples/s]

Spinning up 8 DataLoader workers...
Data ready for LoRA Fine-Tuning.


## The Fine-Tuning Loop

In [15]:
if FINE_TUNE_MODEL:
    # Create an optimizer that ONLY tracks the trainable parameters
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4) # LoRA usually tolerates slightly higher LRs

    # Run your highly optimized training loop
    train_losses, val_losses = train_efficient(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        config=config,
        device=device,
        eval_every=100,
        num_epochs=1,
        max_steps=500, # 500 update steps should be enough to see a behavioral shift
        accumulation_steps=8
    )

    # Save the adapter weights!
    torch.save(model.state_dict(), LORA_SAVE_PATH)
    print("LoRA Adaptation Complete and Saved.")

Epoch 0 | Update Step 100 | Train loss: 5.6519 | Val loss: 5.7265
Epoch 0 | Update Step 200 | Train loss: 5.3916 | Val loss: 5.2995
Epoch 0 | Update Step 300 | Train loss: 5.2566 | Val loss: 5.1006
Epoch 0 | Update Step 400 | Train loss: 5.1540 | Val loss: 4.9574
Epoch 0 | Update Step 500 | Train loss: 5.0432 | Val loss: 4.8341
Reached max_steps (500). Stopping training early.
LoRA Adaptation Complete and Saved.


## Load LoRA Model

In [19]:
# 1. Initialize a fresh base model
adapted_model = GPTModel(CONFIG).to(device)

# 2. Inject the LoRA wrappers so the architecture perfectly matches what you saved
replace_linears_with_lora(adapted_model, rank=8, alpha=16, name_filter=target_attention_layers)

# 3. Load the raw state dictionary from the file
raw_state_dict = torch.load(LORA_SAVE_PATH, map_location=device)

# Create a new dictionary to hold the cleaned keys
clean_state_dict = {}
for key, value in raw_state_dict.items():
    # If the key has the compiler prefix, slice off the first 10 characters ("_orig_mod.")
    if key.startswith("_orig_mod."):
        clean_state_dict[key[10:]] = value
    else:
        clean_state_dict[key] = value

# Load the cleaned dictionary!
adapted_model.load_state_dict(clean_state_dict)

# 4. Set to evaluation mode for text generation
adapted_model.eval()
print("Fine-tuned model successfully loaded!")

Fine-tuned model successfully loaded!


## Output Evaluation

In [20]:
# Test generation to evaluate adapted model against the baseline
prompts = [
    "Item 1A. Risk Factors. As a wholesale power trading entity, our financial condition is highly dependent on grid stability and commodity price fluctuations. During periods of extreme weather, unexpected margin calls from our clearing broker could result in",
    "Management's Discussion and Analysis: Regarding the implied volatility skew observed in our equity derivatives portfolio, particularly the out-of-the-money NVDA call options, the primary driver of this pricing anomaly is",
    """Extract the core financial risk from the following paragraph:
Paragraph: The ongoing litigation regarding the delayed deployment of our high-performance computing clusters has severely restricted our operating cash flow, forcing us to draw $50 million from our revolving credit facility.
Core Risk:""",
    "On March 15, the Board of Directors announced the immediate resignation of the Chief Financial Officer. To ensure continuity in our algorithmic trading division and to oversee the upcoming quarterly audit, the Board has appointed",
]

print("--- Adapted (LoRA) Model Text Generation ---")

# Ensure the model is in evaluation mode
model.eval()

for p in prompts:
    # Use 'encoding' (Tiktoken) to match the fine-tuning dataset processing
    input_ids = encoding.encode(p)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # Generate 100 tokens per prompt to match the baseline comparison
    output_tensor = generate_text(model, input_tensor, max_new_tokens=100)

    # Decode the tokens back to text
    generated_text = encoding.decode(output_tensor[0].tolist())

    print(f"\nPrompt: {p}")
    print(f"Output: {generated_text}")

--- Adapted (LoRA) Model Text Generation ---


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(



Prompt: Item 1A. Risk Factors. As a wholesale power trading entity, our financial condition is highly dependent on grid stability and commodity price fluctuations. During periods of extreme weather, unexpected margin calls from our clearing broker could result in
Output: Item 1A. Risk Factors. As a wholesale power trading entity, our financial condition is highly dependent on grid stability and commodity price fluctuations. During periods of extreme weather, unexpected margin calls from our clearing broker could result in a loss of adverse tax or other regulatory changes in the future to its stock or other investors, which will adversely affect results of operations and financial position.
COMPETITION

In addition to a variety of factors, including the relative size of its business strategy and the use of new market opportunities that may materially reduce rates on favorable terms and the market prices for customers to the Company's and international trading businesses are also affect